In [1]:
import pandas as pd
import numpy as np

def find_best_configs(csv_path1, csv_path2, csv_path3, top_k=10):
    # 1. Load the two CSV files
    df1 = pd.read_csv(csv_path1)
    df2 = pd.read_csv(csv_path2)
    df3 = pd.read_csv(csv_path3)
    
    # 2. Combine datasets
    df = pd.concat([df1, df2, df3], ignore_index=True)
    df.columns = [c.strip() for c in df.columns]

    # 3. Preprocessing: focus on source-to-target alignment
    #df = df[df['direction'] == 'src→tgt'].copy()
    
    metrics = ['mean_cosine', 'top1_acc', 'mean_rank', 'structure_preservation']
    for m in metrics:
        # Convert to numeric, handling potential comma decimals
        df[m] = pd.to_numeric(df[m].astype(str).str.replace(',', '.'), errors='coerce')

    # 4. Aggregate by configuration to find robust winners across both CSVs
    # This finds the "best team" that performs well on average.
    summary = df.groupby('configuration_id')[metrics].mean().reset_index()

    # 5. Normalization for the Combined Score
    # Mean Rank: Lower is better, so we normalize between 0 and 1
    r_min, r_max = summary['mean_rank'].min(), summary['mean_rank'].max()
    summary['rank_norm'] = (summary['mean_rank'] - r_min) / (r_max - r_min) if r_max != r_min else 0

    # 6. Calculate Combined Score (Lower is better)
    # Formula: 0.25*NormRank + 0.25*(1-Cos) + 0.25*(1-Struct) + 0.25*(1-Acc)
    summary['combined_score'] = (
        0.25 * summary['rank_norm'] +
        0.25 * (1 - summary['mean_cosine']) +
        0.25 * (1 - summary['structure_preservation']) +
        0.25 * (1 - summary['top1_acc'])
    )

    # 7. Rank and return top results
    return summary.sort_values('combined_score').head(top_k)

# Example usage:
best_team = find_best_configs('/home/pallucchini/NEW_ALIGNMENT_METHOD/align_new/ablation_study/grid_search_hierarchical_weights_gra_gte/grid/grid_results.csv', '/home/pallucchini/NEW_ALIGNMENT_METHOD/align_new/ablation_study/grid_search_hierarchical_weights_qwen_gtel/grid/grid_results.csv', '/home/pallucchini/NEW_ALIGNMENT_METHOD/align_new/ablation_study/grid_search_hierarchical_weights_gtr_e5/grid/grid_results.csv')
# print(best_team.to_string(index=False))

In [2]:
best_team

,configuration_id,mean_cosine,top1_acc,mean_rank,structure_preservation,rank_norm,combined_score
24,MACRO_STR | SimTlm + StrMain,0.930685,1.0,1.0,0.908565,0.0,0.040188
21,MACRO_STR | SimTlm + StrEq,0.941172,1.0,1.0,0.895715,0.0,0.040778
20,MACRO_STR | SimEq + StrMain,0.944103,1.0,1.0,0.889207,0.0,0.041673
22,MACRO_STR | SimTlm + StrGlo,0.946196,1.0,1.0,0.884796,0.0,0.042252
16,MACRO_STR | SimCon + StrMain,0.948977,1.0,1.0,0.878183,0.0,0.043210
17,MACRO_STR | SimEq + StrEq,0.954098,1.0,1.0,0.871694,0.0,0.043552
18,MACRO_STR | SimEq + StrGlo,0.959254,1.0,1.0,0.859199,0.0,0.045387
19,MACRO_STR | SimEq + StrLoc,0.959074,1.0,1.0,0.858424,0.0,0.045626
13,MACRO_STR | SimCon + StrEq,0.959079,1.0,1.0,0.855497,0.0,0.046356
0,Baseline,0.961625,1.0,1.0,0.844579,0.0,0.048449


In [4]:
import pandas as pd
import numpy as np

csv_path1 = '/home/pallucchini/NEW_ALIGNMENT_METHOD/align_new/ablation_study/grid_search_hierarchical_weights_gra_gte/grid/grid_results.csv'
csv_path2 = '/home/pallucchini/NEW_ALIGNMENT_METHOD/align_new/ablation_study/grid_search_hierarchical_weights_qwen_gtel/grid/grid_results.csv'
csv_path3 = '/home/pallucchini/NEW_ALIGNMENT_METHOD/align_new/ablation_study/grid_search_hierarchical_weights_gtr_e5/grid/grid_results.csv'

df1 = pd.read_csv(csv_path1)
df2 = pd.read_csv(csv_path2)
df3 = pd.read_csv(csv_path3)

# 2. Combine datasets
df = pd.concat([df1, df2, df3], ignore_index=True)
df.columns = [c.strip() for c in df.columns]

# 1. Copia i dati originali (assumendo che ogni riga sia una coppia src->tgt)
df_z = df.copy()

# 2. Configurazione Metriche
metrics = ['mean_rank', 'structure_preservation', 'mean_cosine', 'top1_acc']

# 3. Calcolo Z-Score "Per Riga" (su tutto il dataset)
# Normalizziamo ogni esperimento rispetto alla media globale
for col in metrics:
    mean_val = df_z[col].mean()
    std_val = df_z[col].std()
    
    if std_val == 0:
        df_z[f'Z_{col}'] = 0
    else:
        df_z[f'Z_{col}'] = (df_z[col] - mean_val) / std_val

# 4. Inversione Mean Rank (Basso è meglio -> Z positivo)
df_z['Z_mean_rank'] = df_z['Z_mean_rank'] * -1

# 5. Calcolo Z-Score Totale per la singola coppia
# Questo rappresenta la qualità di QUELLO specifico esperimento src->tgt
df_z['PAIR_Z_SCORE'] = (
    df_z['Z_mean_rank'] + 
    df_z['Z_structure_preservation'] + 
    df_z['Z_mean_cosine'] + 
    df_z['Z_top1_acc']
)

# 6. Aggregazione per Configurazione: Media e Varianza tra le coppie
# Qui otteniamo la risposta alla tua richiesta:
# - mean: "Quanto è brava in media?"
# - std: "Quanto varia tra una coppia e l'altra?"
grouped = df_z.groupby(['configuration_id'])['PAIR_Z_SCORE'].agg(['mean', 'std', 'count'])

# 7. Creazione DataFrame Finale
results = pd.DataFrame()
results['configuration'] = grouped.index
results['Z_Mean'] = grouped['mean'].values  # Media degli Z-score
results['Z_Std'] = grouped['std'].values    # Varianza (Std Dev) degli Z-score
results['Num_Pairs'] = grouped['count'].values

# 8. Ordinamento: Preferiamo Media Alta e (a parità) Stabilità (Bassa Std)
# Possiamo creare uno score sintetico "Conservative": Media - (0.5 * Std)
# Questo penalizza le configurazioni instabili.
results['Conservative_Score'] = results['Z_Mean'] - (0.5 * results['Z_Std'].fillna(0))

results = results.sort_values(by='Z_Mean', ascending=False).head(10)

# Formattazione per la stampa
print("=== TOP 10 CONFIGURAZIONI (Stabilità & Performance) ===")
print("Z_Mean: Performance Media | Z_Std: Instabilità (Varianza)")
print("-" * 80)
print(results[['configuration', 'Z_Mean', 'Z_Std', 'Conservative_Score']].to_string(
    index=False, 
    formatters={
        'Z_Mean': '{:.3f}'.format,
        'Z_Std': '{:.3f}'.format,
        'Conservative_Score': '{:.3f}'.format
    }
))

=== TOP 10 CONFIGURAZIONI (Stabilità & Performance) ===
Z_Mean: Performance Media | Z_Std: Instabilità (Varianza)
--------------------------------------------------------------------------------
               configuration Z_Mean Z_Std Conservative_Score
  MACRO_STR | SimEq + StrGlo  0.687 0.946              0.214
 MACRO_STR | SimCon + StrLoc  0.673 0.844              0.251
 MACRO_STR | SimCon + StrGlo  0.668 0.850              0.243
  MACRO_STR | SimEq + StrLoc  0.662 0.872              0.227
  MACRO_STR | SimCon + StrEq  0.615 0.990              0.120
                    Baseline  0.602 0.883              0.161
   MACRO_STR | SimEq + StrEq  0.555 0.953              0.079
  MACRO_SIM | SimTlm + StrEq  0.436 0.908             -0.018
MACRO_SIM | SimTlm + StrMain  0.328 1.162             -0.253
MACRO_STR | SimCon + StrMain  0.328 1.171             -0.258
